In [1]:
#!/usr/bin/env python3
"""
vllm_pipeline.py -- Neuro-Symbolic Pipeline with vLLM (Batch Inference)

EXACT 2026 -- XAI Challenge @ IJCNN
Qwen2.5-7B + Z3 + vLLM | Kaggle T4x2, No Cloud API

Speedup 5-10x so voi ban transformers nho:
  - vLLM continuous batching + PagedAttention
  - Tensor Parallelism (2x T4 GPU)
  - Xu ly ALL samples cung luc thay vi tung cai mot

Pipeline 5 giai doan:
  Stage 0: Install & Config
  Stage 1: Load vLLM Engine
  Stage 2: Data Grounding + Dual-Layer Ontology
  Stage 3: Z3 Compiler (deterministic, no AI)
  Stage 4: Batch Pipeline (Formalize -> Z3 -> Correct -> Answers)
  Stage 5: Evaluation & Export

Copy-paste vao Kaggle Notebook va chay tung cell.
"""


'\nvllm_pipeline.py -- Neuro-Symbolic Pipeline with vLLM (Batch Inference)\n\nEXACT 2026 -- XAI Challenge @ IJCNN\nQwen2.5-7B + Z3 + vLLM | Kaggle T4x2, No Cloud API\n\nSpeedup 5-10x so voi ban transformers nho:\n  - vLLM continuous batching + PagedAttention\n  - Tensor Parallelism (2x T4 GPU)\n  - Xu ly ALL samples cung luc thay vi tung cai mot\n\nPipeline 5 giai doan:\n  Stage 0: Install & Config\n  Stage 1: Load vLLM Engine\n  Stage 2: Data Grounding + Dual-Layer Ontology\n  Stage 3: Z3 Compiler (deterministic, no AI)\n  Stage 4: Batch Pipeline (Formalize -> Z3 -> Correct -> Answers)\n  Stage 5: Evaluation & Export\n\nCopy-paste vao Kaggle Notebook va chay tung cell.\n'

In [2]:
# ==================================================================
# FIX cho Kaggle T4 -- CELL ĐẦU TIÊN, CHẠY TRƯỚC TẤT CẢ
# ==================================================================
import os, shutil, glob

# --- Fix 1: Symlink libcuda.so ---
STUB_DIR = "/tmp/cuda_stubs"
os.makedirs(STUB_DIR, exist_ok=True)
stub = os.path.join(STUB_DIR, "libcuda.so")

# Xóa link cũ
if os.path.exists(stub) or os.path.islink(stub):
    os.remove(stub)

# Tìm libcuda.so.1 thực tế trên Kaggle
for candidate in [
    "/usr/lib/x86_64-linux-gnu/libcuda.so.1",
    "/usr/lib/x86_64-linux-gnu/libcuda.so",
    *glob.glob("/usr/**/libcuda.so*", recursive=True),
]:
    if os.path.exists(candidate) and not os.path.islink(candidate):
        os.symlink(candidate, stub)
        print(f"✓ Symlink: {stub} -> {candidate}")
        break
else:
    # Fallback: tìm bằng ldconfig
    import subprocess
    result = subprocess.run(["ldconfig", "-p"], capture_output=True, text=True)
    for line in result.stdout.splitlines():
        if "libcuda.so" in line:
            path = line.split("=>")[-1].strip()
            if os.path.exists(path):
                os.symlink(path, stub)
                print(f"✓ Symlink (ldconfig): {stub} -> {path}")
                break

# --- Fix 2: QUAN TRỌNG - Set CẢ compile-time VÀ runtime path ---
os.environ["LIBRARY_PATH"] = f"{STUB_DIR}:" + os.environ.get("LIBRARY_PATH", "")
os.environ["LD_LIBRARY_PATH"] = f"{STUB_DIR}:" + os.environ.get("LD_LIBRARY_PATH", "")

# --- Fix 3: Xóa cache FlashInfer bị lỗi ---
shutil.rmtree("/root/.cache/flashinfer", ignore_errors=True)

# --- Fix 4: Verify ---
print(f"LIBRARY_PATH    = {os.environ.get('LIBRARY_PATH', '')[:80]}...")
print(f"LD_LIBRARY_PATH = {os.environ.get('LD_LIBRARY_PATH', '')[:80]}...")
print(f"Stub exists     = {os.path.exists(stub)}")
if os.path.exists(stub):
    print(f"Stub target     = {os.path.realpath(stub)}")
print("\n✓ Kaggle T4 fixes applied!")

✓ Symlink: /tmp/cuda_stubs/libcuda.so -> /usr/local/nvidia/lib64/libcuda.so.580.105.08
LIBRARY_PATH    = /tmp/cuda_stubs:/usr/local/cuda/lib64/stubs...
LD_LIBRARY_PATH = /tmp/cuda_stubs:/usr/local/nvidia/lib64:/usr/local/cuda/lib64:/usr/local/cuda/li...
Stub exists     = True
Stub target     = /usr/local/nvidia/lib64/libcuda.so.580.105.08

✓ Kaggle T4 fixes applied!


In [3]:

# ==================================================================
# STAGE 0 -- Install Dependencies & Config
# ==================================================================

import subprocess, sys

pkgs = [
    "vllm",
    "z3-solver",
]
print("Installing vLLM + z3-solver (co the mat 2-5 phut)...")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "--quiet", "--break-system-packages"]
    + pkgs,
    check=True,
)
print("All packages installed OK")

import json, os, re, time, csv, traceback
from pathlib import Path
from dataclasses import dataclass, field

import torch
from vllm import LLM, SamplingParams
from transformers import AutoTokenizer
from z3 import (
    Int, IntSort, IntVal, BoolSort, Function,
    ForAll, Exists, And, Or, Not, Implies, Solver, sat, unsat,
)

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA OK  : {torch.cuda.is_available()}")
N_GPUS = torch.cuda.device_count()
for i in range(N_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"GPU {i}    : {props.name}  ({props.total_memory / 1024**3:.1f} GB)")
print("Imports OK")


Installing vLLM + z3-solver (co the mat 2-5 phut)...
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 248.2/248.2 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 82.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.2/295.2 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 26.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 MB 33.9 MB/s eta 0:00:00
   

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.35.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
ydata-profiling 4.18.1 requires numba<0.63,>=0.60, but you have numba 0.65.0 which is incompatible.
pylibcudf-cu12 26.2.1 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.2.0 which is incompatible.
libcuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,cusparse]==12.*, but you have cuda-toolkit 13.0.2 which is incompatible.
rmm-cu12 26.2.0 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.2.0 which is incompatible.
cuml-cu12 26.2.0 requires cuda-python<13.0,>=12.9.2, but you have cuda-python 13.2.0 which is incompatible.
cuml-cu12 26.2.0 requires cuda-toolkit[cublas,cufft,curand,cusolver,

All packages installed OK
PyTorch  : 2.11.0+cu130
CUDA OK  : True
GPU 0    : Tesla T4  (14.6 GB)
GPU 1    : Tesla T4  (14.6 GB)
Imports OK


In [5]:

# ==================================================================
# CAU HINH -- Chinh sua o day
# ==================================================================
QWEN_MODEL_ID  = "Qwen/Qwen2.5-7B-Instruct"
DATASET_PATH   = "/kaggle/input/datasets/nguyenminhtric/test-pipeline/Logic_Based_Educational_Queries (2).json"
PHYSICS_DATASET_PATH = "/kaggle/input/datasets/nguyenminhtric/test-pipeline/Physics_Problems.csv"
N_SAMPLES      = 411
MAX_RETRIES    = 3
OUTPUT_PATH    = "/kaggle/working/pipeline_results_vllm.json"
MAX_NEW_TOKENS = 4080      # token toi da cho formalization
ANS_MAX_TOKENS = 600       # token toi da cho answer extraction

# vLLM Config
TENSOR_PARALLEL  = min(N_GPUS, 2)  # Tu dong dung so GPU co san
MAX_MODEL_LEN    = 8192            # Giam tu 32k de tiet kiem KV cache
GPU_MEMORY_UTIL  = 0.90            # % VRAM cho vLLM
DTYPE            = "half"          # fp16 cho T4 (khong ho tro bf16)
# ==================================================================

print(f"Config OK")
print(f"  Model         : {QWEN_MODEL_ID}")
print(f"  Tensor Parallel: {TENSOR_PARALLEL} GPU(s)")
print(f"  Max Model Len : {MAX_MODEL_LEN}")
print(f"  N_SAMPLES     : {N_SAMPLES}  MAX_RETRIES: {MAX_RETRIES}")



Config OK
  Model         : Qwen/Qwen2.5-7B-Instruct
  Tensor Parallel: 2 GPU(s)
  Max Model Len : 8192
  N_SAMPLES     : 411  MAX_RETRIES: 3


In [6]:

# ==================================================================
# STAGE 1 -- Load vLLM Engine
# ==================================================================

print(f"\nLoading vLLM engine ({QWEN_MODEL_ID})...")
print("  (Lan dau tai ~15 GB tu HuggingFace, sau do cache)")

t_load = time.time()

llm = LLM(
    model=QWEN_MODEL_ID,
    tensor_parallel_size=TENSOR_PARALLEL,
    dtype=DTYPE,
    max_model_len=MAX_MODEL_LEN,
    gpu_memory_utilization=GPU_MEMORY_UTIL,
    trust_remote_code=True,
    enforce_eager=False,       # Dung CUDA graph de tang toc
)

tokenizer = AutoTokenizer.from_pretrained(
    QWEN_MODEL_ID, trust_remote_code=True
)

t_loaded = time.time() - t_load
print(f"vLLM Engine loaded in {t_loaded:.1f}s")
print("OK")




Loading vLLM engine (Qwen/Qwen2.5-7B-Instruct)...
  (Lan dau tai ~15 GB tu HuggingFace, sau do cache)
INFO 05-23 03:48:44 [utils.py:240] non-default args: {'trust_remote_code': True, 'dtype': 'half', 'max_model_len': 8192, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.9, 'disable_log_stats': True, 'model': 'Qwen/Qwen2.5-7B-Instruct'}


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

INFO 05-23 03:49:01 [model.py:568] Resolved architecture: Qwen2ForCausalLM
WARNING 05-23 03:49:01 [model.py:2035] Casting torch.bfloat16 to torch.float16.
INFO 05-23 03:49:01 [model.py:1697] Using max model len 8192
INFO 05-23 03:49:01 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 05-23 03:49:01 [vllm.py:886] Asynchronous scheduling is enabled.
INFO 05-23 03:49:01 [kernel.py:212] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

(Worker_TP0 pid=243) INFO 05-23 03:51:24 [backends.py:393] Compiling a graph for compile range (1, 8192) takes 12.90 s
(Worker_TP0 pid=243) INFO 05-23 03:51:37 [decorators.py:708] saved AOT compiled function to /root/.cache/vllm/torch_compile_cache/torch_aot_compile/b226de5531942cc64b75b46eecc5ca3923bfc087df895194ebdb4db06bd5cb27/rank_0_0/model
(Worker_TP0 pid=243) INFO 05-23 03:51:37 [monitor.py:53] torch.compile took 35.13 s in total
(Worker_TP0 pid=243) INFO 05-23 03:51:40 [monitor.py:81] Initial profiling/warmup run took 3.40 s
(EngineCore pid=219) INFO 05-23 03:52:02 [shm_broadcast.py:681] No available shared memory broadcast block found in 60 seconds. This typically happens when some processes are hanging or doing some time-consuming work (e.g. compilation, weight/kv cache quantization).
(EngineCore pid=219) INFO 05-23 03:53:02 [shm_broadcast.py:681] No available shared memory broadcast block found in 60 seconds. This typically happens when some processes are hanging or doing som

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:05<00:00,  9.02it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 12.20it/s]


(Worker_TP1 pid=244) INFO 05-23 03:55:05 [custom_all_reduce.py:215] Registering 4902 cuda graph addresses
(Worker_TP0 pid=243) INFO 05-23 03:55:05 [custom_all_reduce.py:215] Registering 4902 cuda graph addresses
(Worker_TP1 pid=244) INFO 05-23 03:55:05 [gpu_worker.py:621] CUDA graph pool memory: 0.79 GiB (actual), 1.48 GiB (estimated), difference: 0.7 GiB (88.3%).
(Worker_TP0 pid=243) INFO 05-23 03:55:05 [gpu_model_runner.py:6243] Graph capturing finished in 10 secs, took 0.79 GiB
(Worker_TP0 pid=243) INFO 05-23 03:55:05 [gpu_worker.py:621] CUDA graph pool memory: 0.79 GiB (actual), 1.48 GiB (estimated), difference: 0.7 GiB (88.3%).
(Worker_TP1 pid=244) INFO 05-23 03:55:05 [jit_monitor.py:54] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
(Worker_TP0 pid=243) INFO 05-23 03:55:05 [jit_monitor.py:54] Kernel JIT monitor activated — Triton JIT compilations during inference will be logged as warnings.
(EngineCore pid=219) INFO 05-23 03:55

(EngineCore pid=219) Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.


(EngineCore pid=219) INFO 05-23 03:55:07 [vllm.py:886] Asynchronous scheduling is enabled.
(EngineCore pid=219) INFO 05-23 03:55:07 [kernel.py:212] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'])
vLLM Engine loaded in 385.4s
OK


In [7]:

# ==================================================================
# STAGE 2 -- Dual-Layer Ontology & Data Loading
# ==================================================================

GLOBAL_ONTOLOGY = {
    "quantifiers": ["forall", "exists"],
    "logical_operators": ["and", "or", "implies", "iff", "not"],
    "ast_node_types": ["quantifier", "connective", "predicate", "variable", "constant"],
}

GLOBAL_ONTOLOGY_TEXT = """
## GLOBAL ONTOLOGY -- BAT BUOC TUAN THU (KHONG duoc sua doi)

### Luong tu (Quantifiers):
  forall  -> forall  (voi moi)
  exists  -> exists  (ton tai)

### Toan tu logic (Logical Operators):
  and     -> AND
  or      -> OR
  implies -> IMPLIES (keo theo)
  iff     -> IFF (tuong duong)
  not     -> NOT (phu dinh)

### So do 4 loai AST Node (phai dung DUNG nhu duoi):
  quantifier : { "type":"quantifier",  "operator":"forall|exists",
                 "bound_variables":["x",...], "body":{...} }
  connective : { "type":"connective",  "operator":"and|or|implies|iff|not",
                 "operands":[{...},{...},...] }
  predicate  : { "type":"predicate",   "name":"PredicateName",
                 "arguments":["x","y",...] }
  variable   : { "type":"variable",    "name":"x" }
  constant   : { "type":"constant",    "name":"SomeName" }

### QUY TAC CUNG (vi pham -> Z3 loi):
  1. Chi dung 4 node type tren, KHONG sang tao them
  2. 'not' chi co DUNG 1 operand
  3. 'implies' co DUNG 2 operands (ve trai, ve phai)
  4. bound_variables phai la list (du chi 1 bien)
  5. Bien dung: x, y, z (lowercase); hang so: PascalCase
"""

# -- Prompt Templates --

FORMALIZATION_SYSTEM = (
    "Ban la chuyen gia logic hinh thuc (FOL). Nhiem vu:\n"
    "  Buoc 1: Tao LOCAL ONTOLOGY -- danh sach cac Predicate quan trong\n"
    "  Buoc 2: Chuyen TUNG tien de thanh cay AST JSON de quy\n\n"
    + GLOBAL_ONTOLOGY_TEXT
    + "\n"
    "Output JSON THUAN TUY (khong markdown, khong text thua):\n"
    '{\n'
    '  "step1_local_ontology": [\n'
    '    {"predicate": "TenPredicate", "arity": 1, "description": "Mo ta ngan"}\n'
    '  ],\n'
    '  "step2_premises_ast": [\n'
    '    {"premise_id": 0, "source_nl": "...", "ast": { <AST node> }}\n'
    '  ]\n'
    '}\n'
)

CORRECTION_SYSTEM = (
    "Ban la chuyen gia sua loi FOL. He thong Z3 da phat hien loi.\n"
    "Nhiem vu: sua lai TOAN BO (Buoc 1 + Buoc 2) de khong con loi compile.\n\n"
    + GLOBAL_ONTOLOGY_TEXT
    + "\n"
    "Loi hay gap:\n"
    "  - Arity khong nhat quan\n"
    "  - Variable chua khai bao trong bound_variables\n"
    "  - Predicate khong co trong Local Ontology (hallucination)\n"
    "  - 'not' co nhieu hon 1 operand\n"
    "  - 'implies' khong du 2 operands\n\n"
    "Output JSON thuan tuy -- format GIONG HET lan dau.\n"
)

ANSWER_SYSTEM = (
    "Ban la chuyen gia suy luan logic.\n"
    "Dua vao cac tien de FOL da duoc xac minh boi Z3, hay tra loi cau hoi.\n\n"
    "Quy tac:\n"
    "  - Trac nghiem (A/B/C/D): tra ve 1 chu cai HOA\n"
    '  - Yes/No: tra ve "Yes", "No", hoac "Unknown"\n'
    "  - Suy luan chat che, khong suy doan ngoai pham vi\n\n"
    "Output JSON THUAN TUY:\n"
    '{"answer": "A|B|C|D|Yes|No|Unknown", "reasoning": "giai thich ngan"}\n'
)

print("Prompt templates san sang")


# -- Dataset Loader --

def load_dataset(path, is_physics=False, max_samples=N_SAMPLES):
    """Load va chuan hoa dataset, ho tro JSON va CSV."""
    if not path or not os.path.exists(path):
        print(f"[WARN] Dataset path not found: {path}")
        return []
    if path.endswith(".csv"):
        with open(path, encoding="utf-8") as f:
            raw = list(csv.DictReader(f))
    else:
        with open(path, encoding="utf-8") as f:
            raw = json.load(f)
    out = raw[:max_samples]
    if is_physics and out:
        for s in out:
            if "premises-NL" not in s:
                s["premises-NL"] = [s.get("question", "")]
            if "questions" not in s:
                s["questions"] = [s.get("question", "")]
            if "answers" not in s:
                s["answers"] = [str(s.get("answer", s.get("final_answer", "Unknown")))]
    return out


# -- JSON Parser --

def safe_json(text):
    """Trich xuat JSON tu response -- robust multi-strategy parser."""
    text = text.strip()
    # 1) Direct
    try:
        return json.loads(text)
    except Exception:
        pass
    # 2) Code block
    m = re.search(r"```(?:json)?\s*\n?(.*?)```", text, re.DOTALL)
    if m:
        try:
            return json.loads(m.group(1).strip())
        except Exception:
            pass
    # 3) Brace matching
    start = text.find("{")
    if start >= 0:
        depth = 0
        end = start
        for i in range(start, len(text)):
            if text[i] == "{":
                depth += 1
            elif text[i] == "}":
                depth -= 1
                if depth == 0:
                    end = i
                    break
        try:
            return json.loads(text[start : end + 1])
        except Exception:
            pass
    return {}


# -- Batch Generate Helper --

def batch_generate(prompt_pairs, max_tokens=MAX_NEW_TOKENS):
    """
    Batch generate using vLLM.

    Args:
        prompt_pairs: list of (system_msg, user_msg) tuples
        max_tokens: max new tokens per response

    Returns:
        list of response strings
    """
    # Format prompts using chat template
    formatted = []
    for sys_msg, usr_msg in prompt_pairs:
        messages = [
            {"role": "system", "content": sys_msg},
            {"role": "user", "content": usr_msg},
        ]
        text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        formatted.append(text)

    params = SamplingParams(
        temperature=0.05,
        max_tokens=max_tokens,
        repetition_penalty=1.1,
    )

    outputs = llm.generate(formatted, params)

    # Sort by request_id to preserve order
    outputs_sorted = sorted(outputs, key=lambda x: int(x.request_id))
    return [o.outputs[0].text.strip() for o in outputs_sorted]


print("batch_generate() san sang")



Prompt templates san sang
batch_generate() san sang


In [8]:

# ==================================================================
# STAGE 3 -- Z3 Compiler (Deterministic, no AI)
# ==================================================================

_func_cache = {}


def get_z3_func(name, arity):
    key = f"{name}_{arity}"
    if key not in _func_cache:
        sorts = [IntSort()] * arity + [BoolSort()]
        _func_cache[key] = Function(name, *sorts)
    return _func_cache[key]


def _resolve_bound_var_name(bv):
    if isinstance(bv, dict):
        return bv.get("name", str(bv))
    return str(bv)


def _resolve_predicate_arg(a, var_map):
    if isinstance(a, str):
        if a in var_map:
            return var_map[a]
        return IntVal(abs(hash(a)) % 100000)
    if isinstance(a, dict):
        atype = a.get("type", "")
        name = a.get("name", "")
        if atype == "variable":
            if name in var_map:
                return var_map[name]
            v = Int(name)
            var_map[name] = v
            return v
        if atype == "constant":
            if name in var_map:
                return var_map[name]
            return IntVal(abs(hash(name)) % 100000)
        raise ValueError(f"Argument khong hop le (type={atype!r})")
    return IntVal(abs(hash(str(a))) % 100000)


def compile_ast(node, var_map):
    """Bien dich 1 AST node -> Z3 expression."""
    if not isinstance(node, dict):
        raise ValueError(f"Expected dict, got {type(node)}: {node!r}")

    ntype = node.get("type", "")

    if ntype == "quantifier":
        op = node.get("operator", "").lower()
        bvs = node.get("bound_variables", [])
        if not bvs:
            raise ValueError("quantifier thieu bound_variables")
        bv_names = [_resolve_bound_var_name(bv) for bv in bvs]
        z3_bvs = [Int(v) for v in bv_names]
        child_map = {**var_map, **{v: z3_bvs[i] for i, v in enumerate(bv_names)}}
        body = compile_ast(node["body"], child_map)
        if op == "forall":
            return ForAll(z3_bvs, body)
        elif op in ("exists", "exist"):
            return Exists(z3_bvs, body)
        else:
            raise ValueError(f"Quantifier khong hop le: {op!r}")

    elif ntype == "connective":
        op = node.get("operator", "").lower()
        ops = [compile_ast(o, var_map) for o in node.get("operands", [])]
        if op == "and":
            return And(*ops)
        elif op == "or":
            return Or(*ops)
        elif op == "implies":
            if len(ops) != 2:
                raise ValueError(f"implies can 2 operands, nhan {len(ops)}")
            return Implies(ops[0], ops[1])
        elif op == "iff":
            if len(ops) != 2:
                raise ValueError(f"iff can 2 operands, nhan {len(ops)}")
            return And(Implies(ops[0], ops[1]), Implies(ops[1], ops[0]))
        elif op == "not":
            if len(ops) != 1:
                raise ValueError(f"not can 1 operand, nhan {len(ops)}")
            return Not(ops[0])
        else:
            raise ValueError(f"Connective khong hop le: {op!r}")

    elif ntype == "predicate":
        name = node.get("name", "")
        args = node.get("arguments", [])
        if not name:
            raise ValueError('predicate thieu "name"')
        func = get_z3_func(name, len(args))
        z3_args = [_resolve_predicate_arg(a, var_map) for a in args]
        return func(*z3_args)

    elif ntype in ("variable", "constant"):
        name = node.get("name", "")
        if name in var_map:
            return var_map[name]
        if ntype == "constant":
            return IntVal(abs(hash(name)) % 100000)
        v = Int(name)
        var_map[name] = v
        return v

    else:
        raise ValueError(f"AST node type khong hop le: {ntype!r}")


def verify_with_z3(premises_ast):
    """Bien dich toan bo premises AST -> Z3, kiem tra consistency."""
    _func_cache.clear()
    solver = Solver()
    errors = []
    compiled = 0

    for item in premises_ast:
        pid = item.get("premise_id", "?")
        try:
            ast = item.get("ast", {})
            if not ast:
                errors.append(f"Premise {pid}: AST rong")
                continue
            expr = compile_ast(ast, {})
            solver.add(expr)
            compiled += 1
        except Exception as e:
            errors.append(f"Premise {pid}: {str(e)[:250]}")

    if errors:
        return {
            "status": "compile_error",
            "errors": errors,
            "compiled_count": compiled,
            "total_count": len(premises_ast),
        }
    try:
        result = solver.check()
        return {
            "status": str(result),
            "errors": [],
            "compiled_count": compiled,
            "total_count": len(premises_ast),
        }
    except Exception as e:
        return {
            "status": "solver_error",
            "errors": [str(e)],
            "compiled_count": compiled,
            "total_count": len(premises_ast),
        }


def hallucination_check(local_ontology, premises_ast):
    """Kiem tra: moi Predicate trong AST phai co trong Local Ontology."""
    declared = set()
    for item in local_ontology:
        if isinstance(item, dict):
            declared.add(item.get("predicate", ""))

    warnings = []

    def collect_predicates(node, found):
        if not isinstance(node, dict):
            return
        if node.get("type") == "predicate":
            found.add(node.get("name", ""))
        for v in node.values():
            if isinstance(v, dict):
                collect_predicates(v, found)
            elif isinstance(v, list):
                for sub in v:
                    collect_predicates(sub, found)

    for item in premises_ast:
        used = set()
        collect_predicates(item.get("ast", {}), used)
        hallucinated = used - declared - {""}
        if hallucinated:
            warnings.append(
                f"Premise {item.get('premise_id', '?')}: "
                f"predicates not in Ontology -> {hallucinated}"
            )
    return warnings


print("Z3 compiler san sang")



Z3 compiler san sang


In [9]:

# ==================================================================
# STAGE 4 -- Batch Pipeline Orchestrator
# ==================================================================

@dataclass
class PipelineResult:
    sample_id: int
    status: str = "pending"
    z3_status: str = "pending"
    z3_compiled: int = 0
    z3_total: int = 0
    z3_attempts: int = 0
    z3_errors: list = field(default_factory=list)
    local_ontology: list = field(default_factory=list)
    premises_ast: list = field(default_factory=list)
    hallucination_warn: list = field(default_factory=list)
    predicted_answers: list = field(default_factory=list)
    ground_truth: list = field(default_factory=list)
    correct_count: int = 0
    total_questions: int = 0
    time_sec: float = 0.0
    error_log: list = field(default_factory=list)


def _make_formalization_user(sample):
    """Tao user prompt cho formalization."""
    premises = sample["premises-NL"]
    numbered = "\n".join(f"Premise {i+1}: {p}" for i, p in enumerate(premises))
    return (
        "Hay hinh thuc hoa cac tien de sau theo dung quy trinh 2 buoc.\n\n"
        + numbered
        + "\n\nNho:\n"
        "  Buoc 1: khai bao Local Ontology\n"
        "  Buoc 2: sinh cay AST JSON de quy cho tung premise\n"
        "  Chi tra ve JSON thuan tuy."
    )


def _make_correction_user(sample, result):
    """Tao user prompt cho correction (khi Z3 bao loi)."""
    premises = sample["premises-NL"]
    numbered = "\n".join(f"Premise {i+1}: {p}" for i, p in enumerate(premises))
    z3_errors = "\n".join(result.z3_errors[:5]) or "(khong co loi cu the)"
    hall_errs = "\n".join(result.hallucination_warn) if result.hallucination_warn else "(khong co)"
    prev_local = json.dumps(result.local_ontology, ensure_ascii=False, indent=2)

    return (
        "He thong Z3 da phat hien loi khi compile cay AST.\n\n"
        "===== LOI TU Z3 =====\n"
        f"Z3 status: {result.z3_status}\n"
        f"Compiled: {result.z3_compiled} / {result.z3_total}\n\n"
        f"Loi chi tiet:\n{z3_errors}\n\n"
        f"Hallucination:\n{hall_errs}\n\n"
        "===== LOCAL ONTOLOGY CU =====\n"
        f"{prev_local}\n\n"
        "===== PREMISES GOC =====\n"
        f"{numbered}\n\n"
        "Sua lai TOAN BO (Buoc 1 + Buoc 2). Chi tra ve JSON thuan tuy."
    )


def _make_answer_user(sample, fol_context, question, q_idx):
    """Tao user prompt cho answer extraction."""
    premises = sample["premises-NL"]
    p_text = "\n".join(f"P{i+1}: {p}" for i, p in enumerate(premises))
    fol_text = "\n".join(f"FOL P{i+1}: {f}" for i, f in enumerate(fol_context))
    return (
        "## Tien de (Natural Language):\n"
        f"{p_text}\n\n"
        "## Tien de (FOL da xac minh qua Z3):\n"
        f"{fol_text}\n\n"
        f"## Cau hoi {q_idx + 1}:\n"
        f"{question}\n\n"
        "Tra loi JSON thuan tuy."
    )


def _process_formalization(result, raw_response):
    """Parse formalization response va chay Z3."""
    formalization = safe_json(raw_response)
    local_onto = formalization.get("step1_local_ontology", [])
    premises_ast = formalization.get("step2_premises_ast", [])

    result.local_ontology = local_onto
    result.premises_ast = premises_ast

    if not premises_ast:
        result.z3_status = "no_ast"
        result.error_log.append("step2_premises_ast rong")
        return

    hw = hallucination_check(local_onto, premises_ast)
    z3_info = verify_with_z3(premises_ast)

    result.hallucination_warn = hw
    result.z3_status = z3_info["status"]
    result.z3_errors = z3_info.get("errors", [])
    result.z3_compiled = z3_info.get("compiled_count", 0)
    result.z3_total = z3_info.get("total_count", 0)


def run_batch_pipeline(samples, dataset_name="Dataset"):
    """
    Chay toan bo pipeline bang BATCH INFERENCE.

    Flow:
      Round 1: Batch formalization (all N samples)
      Round 2..MAX_RETRIES: Batch correction (only failed samples)
      Final: Batch answer extraction (all questions)
    """
    N = len(samples)
    t0_all = time.time()

    results = [
        PipelineResult(
            sample_id=i,
            ground_truth=samples[i].get("answers", []),
            total_questions=len(samples[i].get("questions", [])),
        )
        for i in range(N)
    ]

    # ===== ROUND 1: Batch Formalization =====
    print(f"\n{'=' * 60}")
    print(f"  [Round 1] Batch Formalization -- {N} samples")
    print(f"{'=' * 60}")

    t1 = time.time()
    form_prompts = [
        (FORMALIZATION_SYSTEM, _make_formalization_user(s))
        for s in samples
    ]
    form_responses = batch_generate(form_prompts, max_tokens=MAX_NEW_TOKENS)
    t1_done = time.time() - t1
    print(f"  vLLM batch done in {t1_done:.1f}s ({t1_done/N:.1f}s/sample)")

    for i, raw in enumerate(form_responses):
        results[i].z3_attempts = 1
        try:
            _process_formalization(results[i], raw)
        except Exception as e:
            results[i].z3_status = "no_ast"
            results[i].error_log.append(f"Round1: {str(e)[:300]}")

    # Stats
    status_counts = {}
    for r in results:
        status_counts[r.z3_status] = status_counts.get(r.z3_status, 0) + 1
    print(f"  Z3 results: {status_counts}")

    # ===== ROUNDS 2..MAX_RETRIES: Batch Correction =====
    for attempt in range(2, MAX_RETRIES + 1):
        # Chi correction cho compile_error (khong phai no_ast)
        failed = [i for i in range(N) if results[i].z3_status == "compile_error"]
        if not failed:
            print(f"  [Round {attempt}] Khong con loi compile -> skip")
            break

        print(f"\n  [Round {attempt}] Batch Correction -- {len(failed)} failed samples")

        t_c = time.time()
        corr_prompts = [
            (CORRECTION_SYSTEM, _make_correction_user(samples[i], results[i]))
            for i in failed
        ]
        corr_responses = batch_generate(corr_prompts, max_tokens=MAX_NEW_TOKENS)
        t_c_done = time.time() - t_c
        print(f"  vLLM batch done in {t_c_done:.1f}s")

        for j, raw in enumerate(corr_responses):
            idx = failed[j]
            results[idx].z3_attempts = attempt
            try:
                _process_formalization(results[idx], raw)
            except Exception as e:
                results[idx].error_log.append(f"Round{attempt}: {str(e)[:300]}")

        # Updated stats
        status_counts = {}
        for r in results:
            status_counts[r.z3_status] = status_counts.get(r.z3_status, 0) + 1
        print(f"  Z3 results: {status_counts}")

    # ===== BATCH ANSWER EXTRACTION =====
    print(f"\n{'=' * 60}")
    print(f"  [Answers] Batch Answer Extraction")
    print(f"{'=' * 60}")

    ans_prompts = []
    ans_mapping = []  # (sample_idx, question_idx)

    for i, s in enumerate(samples):
        fol_ctx = []
        for item in results[i].premises_ast:
            fol_ctx.append(item.get("source_nl", ""))
        for q_idx, q in enumerate(s.get("questions", [])):
            prompt = (ANSWER_SYSTEM, _make_answer_user(s, fol_ctx, q, q_idx))
            ans_prompts.append(prompt)
            ans_mapping.append((i, q_idx))

    if ans_prompts:
        t_a = time.time()
        ans_responses = batch_generate(ans_prompts, max_tokens=ANS_MAX_TOKENS)
        t_a_done = time.time() - t_a
        print(f"  vLLM batch done in {t_a_done:.1f}s ({len(ans_prompts)} questions)")

        for j, raw in enumerate(ans_responses):
            sample_idx, q_idx = ans_mapping[j]
            try:
                ans = safe_json(raw)
                results[sample_idx].predicted_answers.append({
                    "question_id": q_idx,
                    "answer": ans.get("answer", "Unknown"),
                    "reasoning": ans.get("reasoning", ""),
                })
            except Exception:
                results[sample_idx].predicted_answers.append({
                    "question_id": q_idx,
                    "answer": "Unknown",
                    "reasoning": raw[:200],
                })

    # ===== FINALIZE STATUS =====
    for i, r in enumerate(results):
        gt = samples[i].get("answers", [])
        correct = sum(
            1
            for ar in r.predicted_answers
            if ar["question_id"] < len(gt)
            and str(ar["answer"]).strip().upper()
            == str(gt[ar["question_id"]]).strip().upper()
        )
        r.correct_count = correct

        if r.z3_status in ("sat", "unsat", "unknown"):
            r.status = "success"
        elif r.z3_compiled > 0:
            r.status = "partial"
        else:
            r.status = "failed"

        r.time_sec = round(time.time() - t0_all, 2)

    total_time = time.time() - t0_all
    print(f"\n  Pipeline {dataset_name} hoan tat in {total_time:.1f}s")
    print(f"  Trung binh: {total_time / N:.1f}s/sample")

    return results


print("Batch pipeline orchestrator san sang")



Batch pipeline orchestrator san sang


In [ ]:

# ==================================================================
# STAGE 5 -- Evaluation & Export
# ==================================================================

def evaluate(results):
    n = len(results)
    if n == 0:
        return {}
    total_q = sum(r.total_questions for r in results)
    total_ok = sum(r.correct_count for r in results)
    status_ct = {"success": 0, "partial": 0, "failed": 0}
    z3_ct = {"sat": 0, "unsat": 0, "unknown": 0, "compile_error": 0, "solver_error": 0, "no_ast": 0, "other": 0}
    for r in results:
        status_ct[r.status] = status_ct.get(r.status, 0) + 1
        key = r.z3_status if r.z3_status in z3_ct else "other"
        z3_ct[key] += 1
    hall_total = sum(len(r.hallucination_warn) for r in results)
    avg_retries = sum(r.z3_attempts for r in results) / n
    return {
        "n_samples": n,
        "total_questions": total_q,
        "total_correct": total_ok,
        "accuracy": round(total_ok / total_q, 4) if total_q else 0,
        "status_breakdown": status_ct,
        "z3_breakdown": z3_ct,
        "hallucination_warnings": hall_total,
        "avg_z3_retries": round(avg_retries, 2),
    }


def result_to_dict(r):
    return {
        "sample_id": r.sample_id,
        "status": r.status,
        "z3_status": r.z3_status,
        "z3_compiled": r.z3_compiled,
        "z3_total": r.z3_total,
        "z3_attempts": r.z3_attempts,
        "z3_errors": r.z3_errors[:3],
        "hallucination_warns": r.hallucination_warn,
        "local_ontology": r.local_ontology,
        "correct_count": r.correct_count,
        "total_questions": r.total_questions,
        "predicted_answers": [a["answer"] for a in r.predicted_answers],
        "ground_truth": r.ground_truth,
        "per_question": [
            {
                "q_id": a["question_id"],
                "predicted": a["answer"],
                "gt": r.ground_truth[a["question_id"]] if a["question_id"] < len(r.ground_truth) else "?",
                "correct": (
                    str(a["answer"]).upper() == str(r.ground_truth[a["question_id"]]).upper()
                    if a["question_id"] < len(r.ground_truth) else False
                ),
                "reasoning": a.get("reasoning", ""),
            }
            for a in r.predicted_answers
        ],
        "time_sec": r.time_sec,
        "error_log": r.error_log[-2:],
    }


def finalize_and_save(results, output_path, dataset_path, dataset_name="Dataset"):
    """Print summary va luu JSON."""
    if not results:
        print(f"[WARN] No results for {dataset_name}")
        return

    metrics = evaluate(results)
    acc_val = metrics.get("accuracy", 0)

    W = 60
    print("\n" + "=" * W)
    print(f"  {dataset_name.upper()} -- EVALUATION SUMMARY")
    print("=" * W)
    print(f"  Model          : {QWEN_MODEL_ID}")
    print(f"  Engine         : vLLM (TP={TENSOR_PARALLEL})")
    print(f"  Samples        : {metrics.get('n_samples', 0)}")
    print(f"  Total questions: {metrics.get('total_questions', 0)}")
    print(f"  Correct        : {metrics.get('total_correct', 0)}")
    print(f"  Accuracy       : {acc_val:.1%}")
    print("-" * W)
    print("  Pipeline Status:")
    for k, v in metrics.get("status_breakdown", {}).items():
        print(f"    {k:14}: {v:3d}  {'#' * v}")
    print("-" * W)
    print("  Z3 Verification:")
    for k, v in metrics.get("z3_breakdown", {}).items():
        if v > 0:
            print(f"    {k:16}: {v:3d}  {'#' * v}")
    print("-" * W)
    print(f"  Hallucination warns: {metrics.get('hallucination_warnings', 0)}")
    print(f"  Avg Z3 retries     : {metrics.get('avg_z3_retries', 0)}")
    print("=" * W)

    # Per-sample table
    hdr = f"{'ID':>3} | {'Status':>8} | {'Z3':>13} | {'Corr':>6} | {'Retry':>5} | Hall"
    print(hdr)
    print("-" * len(hdr))
    for r in results:
        hall = f"W{len(r.hallucination_warn)}" if r.hallucination_warn else "ok"
        print(
            f"{r.sample_id:>3} | {r.status:>8} | {r.z3_status:>13} | "
            f"{r.correct_count}/{r.total_questions:>4} | {r.z3_attempts:>5} | {hall}"
        )

    # Save JSON
    output_data = {
        "meta": {
            "model": QWEN_MODEL_ID,
            "engine": "vLLM",
            "tensor_parallel": TENSOR_PARALLEL,
            "n_samples": len(results),
            "max_retries": MAX_RETRIES,
            "dataset": dataset_path,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        },
        "metrics": metrics,
        "per_sample": [result_to_dict(r) for r in results],
    }
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(output_data, f, ensure_ascii=False, indent=2)

    fsize = Path(output_path).stat().st_size / 1024
    total_c = metrics.get("total_correct", 0)
    total_q = metrics.get("total_questions", 0)
    print(f"\nKet qua luu tai: {output_path}")
    print(f"  Dung luong: {fsize:.1f} KB")
    print(f"  Final Accuracy: {acc_val:.1%}  ({total_c}/{total_q})")


# ==================================================================
# RUN
# ==================================================================

print("\n" + "=" * 65)
print("  NEURO-SYMBOLIC PIPELINE -- vLLM BATCH INFERENCE")
print(f"  Model: {QWEN_MODEL_ID}  |  TP: {TENSOR_PARALLEL} GPU(s)")
print("=" * 65)

# --- Logic Dataset ---
logic_samples = load_dataset(DATASET_PATH, is_physics=False)
print(f"\nLogic Dataset: {len(logic_samples)} samples")

if logic_samples:
    logic_results = run_batch_pipeline(logic_samples, dataset_name="Logic Dataset")
    finalize_and_save(logic_results, OUTPUT_PATH, DATASET_PATH, "Logic Dataset")

# --- Physics Dataset ---
if PHYSICS_DATASET_PATH:
    physics_samples = load_dataset(PHYSICS_DATASET_PATH, is_physics=True)
    print(f"\nPhysics Dataset: {len(physics_samples)} samples")

    if physics_samples:
        physics_output = OUTPUT_PATH.replace(".json", "_physics.json")
        physics_results = run_batch_pipeline(physics_samples, dataset_name="Physics Dataset")
        finalize_and_save(physics_results, physics_output, PHYSICS_DATASET_PATH, "Physics Dataset")

print("\n" + "=" * 65)
print("  PIPELINE HOAN TAT")
print("=" * 65)



  NEURO-SYMBOLIC PIPELINE -- vLLM BATCH INFERENCE
  Model: Qwen/Qwen2.5-7B-Instruct  |  TP: 2 GPU(s)

Logic Dataset: 411 samples

  [Round 1] Batch Formalization -- 411 samples


Rendering prompts:   0%|          | 0/411 [00:00<?, ?it/s]

(Worker_TP0 pid=243) WARNING 05-23 03:55:10 [jit_monitor.py:103] Triton kernel JIT compilation during inference: _compute_slot_mapping_kernel. This causes a latency spike; consider extending warmup to cover this shape/config.


Processed prompts:   0%|          | 0/411 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]